In [1]:
import json
import re
from collections import defaultdict

myfile = r'C:\Users\sayala\Desktop\delete_9069_system_metadata.json'
with open(myfile, "r") as f:
    data = json.load(f)

metrics = data["Metrics"]

combiner_strings = defaultdict(set)

for key, sensor in metrics.items():

    sensor_name = sensor.get("sensor_name", "")

    match = re.search(
        r"Combiner DC Input_(\d+)\.(\d+)\.(\d+)_DC CURRENT STRING (\d+)",
        sensor_name
    )

    if match:
        group = int(match.group(1))
        combiner = int(match.group(2))
        string = int(match.group(3))

        combiner_id = (group, combiner)

        combiner_strings[combiner_id].add(string)


print("Number of combiner boxes:", len(combiner_strings))

print("\nStrings per combiner:")

for (group, combiner), strings in sorted(combiner_strings.items()):

    strings = sorted(strings)

    print(
        f"{group:02d}.{combiner:02d}: "
        f"{len(strings)} strings "
        f"(strings {min(strings):02d}-{max(strings):02d})"
    )

Number of combiner boxes: 280

Strings per combiner:
01.01: 13 strings (strings 01-13)
01.02: 12 strings (strings 01-12)
01.03: 12 strings (strings 01-12)
01.04: 12 strings (strings 01-12)
01.05: 12 strings (strings 01-12)
01.06: 12 strings (strings 01-12)
01.07: 12 strings (strings 01-12)
02.01: 12 strings (strings 01-12)
02.02: 13 strings (strings 01-13)
02.03: 12 strings (strings 01-12)
02.04: 13 strings (strings 01-13)
02.05: 12 strings (strings 01-12)
02.06: 12 strings (strings 01-12)
02.07: 12 strings (strings 01-12)
03.01: 12 strings (strings 01-12)
03.02: 12 strings (strings 01-12)
03.03: 12 strings (strings 01-12)
03.04: 12 strings (strings 01-12)
03.05: 12 strings (strings 01-12)
03.06: 12 strings (strings 01-12)
03.07: 13 strings (strings 01-13)
04.01: 12 strings (strings 01-12)
04.02: 12 strings (strings 01-12)
04.03: 12 strings (strings 01-12)
04.04: 12 strings (strings 01-12)
04.05: 12 strings (strings 01-12)
04.06: 13 strings (strings 01-13)
04.07: 13 strings (strings 01

In [2]:
all_strings = set()

for key, sensor in metrics.items():

    sensor_name = sensor.get("sensor_name", "")

    match = re.search(
        r"Combiner DC Input_(\d+)\.(\d+)\.(\d+)_DC CURRENT STRING",
        sensor_name
    )

    if match:
        group = int(match.group(1))
        combiner = int(match.group(2))
        string = int(match.group(3))

        # Full identifier uniquely identifies a physical string
        all_strings.add((group, combiner, string))

print("Total number of strings:", len(all_strings))

Total number of strings: 3408


In [3]:
# Collect combiner/string information
by_inverter = defaultdict(lambda: {
    "combiners": set(),
    "strings": set()
})

pattern = re.compile(
    r"Combiner DC Input_(\d+)\.(\d+)\.(\d+)_DC CURRENT STRING"
)

for sensor in data["Metrics"].values():

    name = sensor.get("sensor_name", "")
    match = pattern.search(name)

    if match:
        inverter = int(match.group(1))
        combiner = int(match.group(2))
        string = int(match.group(3))

        by_inverter[inverter]["combiners"].add(combiner)
        by_inverter[inverter]["strings"].add((combiner, string))


# Print comparison
print(
    f"{'Inv':>4} {'Combiners':>10} {'Measured strings':>18} "
    f"{'Metadata strings':>17}"
)
print("-" * 55)

for inverter_num in sorted(by_inverter):

    # 01 corresponds to "Inverter 0", 02 to "Inverter 1", etc.
    inverter_metadata = data["Inverters"].get(
        f"Inverter {inverter_num - 1}", {}
    )

    metadata_strings = inverter_metadata.get("num_strings", "NA")

    n_combiners = len(by_inverter[inverter_num]["combiners"])
    n_strings = len(by_inverter[inverter_num]["strings"])

    print(
        f"{inverter_num:>4} "
        f"{n_combiners:>10} "
        f"{n_strings:>18} "
        f"{metadata_strings:>17}"
    )

 Inv  Combiners   Measured strings  Metadata strings
-------------------------------------------------------
   1          7                 85               272
   2          7                 86               272
   3          7                 85               272
   4          7                 86               272
   5          7                 85               272
   6          7                 86               272
   7          7                 86               272
   8          7                 85               272
   9          7                 86               272
  10          7                 85               272
  11          7                 85               272
  12          7                 86               272
  13          7                 85               272
  14          7                 86               272
  15          7                 88               272
  16          7                 83               272
  17          7                 85         

In [4]:

for inverter_name, inverter in data["Inverters"].items():
    if "AC capacity(kW)" in inverter:
        print(
            inverter_name,
            inverter.get("name"),
            inverter["AC capacity(kW)"]
        )

Inverter 0 sga_inv_1 825
Inverter 1 sga_inv_2 825
Inverter 2 sga_inv_3 825
Inverter 3 sga_inv_4 825
Inverter 4 sga_inv_5 825


In [5]:
for inverter_name, inverter in data["Inverters"].items():
    print("\n", inverter_name)
    for key, value in inverter.items():
        print(f"  {key}: {value}")


 Inverter 0
  inverter_id: 29972
  name: sga_inv_1
  manufacturer: Unknown
  model: 
  type: string
  quantity: 1
  serial_num: 
  comments: 
  time_interval: L
  modules_per_string: 19
  num_strings: 272
  AC capacity(kW): 825
  maximum DC current(A): 1600
  minimum DC voltage(V DC): 545

 Inverter 1
  inverter_id: 29973
  name: sga_inv_2
  manufacturer: Unknown
  model: 
  type: string
  quantity: 1
  serial_num: 
  comments: 
  time_interval: L
  modules_per_string: 19
  num_strings: 272
  AC capacity(kW): 825
  maximum DC current(A): 1600
  minimum DC voltage(V DC): 545

 Inverter 2
  inverter_id: 29974
  name: sga_inv_3
  manufacturer: Unknown
  model: 
  type: string
  quantity: 1
  serial_num: 
  comments: 
  time_interval: L
  modules_per_string: 19
  num_strings: 272
  AC capacity(kW): 825
  maximum DC current(A): 1600
  minimum DC voltage(V DC): 545

 Inverter 3
  inverter_id: 29975
  name: sga_inv_4
  manufacturer: Unknown
  model: 
  type: string
  quantity: 1
  serial_num